In [1]:
!pip install torch transformers accelerate bitsandbytes
!pip install datasets peft sentencepiece wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 13.4 MB/s eta 0:00:00


In [2]:
import torch, math, wandb
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, PeftModel
from tqdm import tqdm
# wandb.init(project="llama2-finetune")

In [3]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_8bit=True,
    device_map="auto"
)
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
def format_example(example):
    disease = example.get("disease", "")
    symptoms = example.get("symptoms", "")
    medicine = example.get("medicine", "")
    precautions = example.get("precautions", "")
    treatment = example.get("treatment", "")
    description = example.get("description", "")

    prompt = (
        f"Disease: {disease}\n"
        f"Symptoms: {symptoms}\n"
        f"Medicine: {medicine}\n"
        f"Precautions: {precautions}\n"
        f"Treatment: {treatment}\n"
        f"Description: {description}"
    ).strip()

    return {"text": prompt}

def tokenize(batch):
    tokenized = tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

device = "cuda" if torch.cuda.is_available() else "cpu"
data = load_dataset("json", data_files="medical_dataset_full.json")
data = data["train"].train_test_split(test_size=0.1, seed=42)
train_data = data["train"].map(format_example)
eval_data = data["test"].map(format_example)
train_tokenized = train_data.map(tokenize, batched=True, remove_columns=["text"])
eval_tokenized = eval_data.map(tokenize, batched=True, remove_columns=["text"])


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [6]:
prompt = "Disease: Influenza"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

In [ ]:
def calculate_perplexity(model, tokenizer, dataset, max_length=512):
    model.eval()
    total_loss = 0
    total_tokens = 0

    for i in tqdm(range(len(dataset))):
        text = dataset[i]['text'].strip()
        if not text:  # skip empty lines
            continue

        inputs = tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        )

        input_ids = inputs["input_ids"]
        if input_ids.numel() == 0:
            continue

        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            loss = model(**inputs, labels=inputs["input_ids"]).loss

        total_loss += loss.item() * input_ids.numel()
        total_tokens += input_ids.numel()

    avg_loss = total_loss / total_tokens
    perplexity = torch.exp(torch.tensor(avg_loss))
    return perplexity.item()

In [7]:
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    repetition_penalty=1.2
)
tokenizer.decode(outputs[0], skip_special_tokens=True)

'Disease: Influenza\nFlu, also known as influenza or seasonal flu, is a contagious viral infection that affects the lungs and can cause severe symptoms. It is caused by type A virus called influenza A (H1N1) or B, depending on the specific strain circulating at any given time. Symptoms of the flu typically include fever, chills, cough, sore throat, runny nose, headache, muscle'

In [ ]:
print(model.config)
print(f"Total parameters: {model.num_parameters()/1e6:.1f}M")
ppl = calculate_perplexity(model, tokenizer, train_data.select(range(200)))
print(f"Perplexity: {ppl:.2f}")

GPT2Config {
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": 50256,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.57.1",
  "use_cache": true,
  "vocab_size": 50257
}

Total parameters: 124.4M


100%|██████████| 200/200 [00:01<00:00, 101.78it/s]


Perplexity: 52.33


### Exact Output of GPT2
What is capital of France? It has to be in one word: Capital, for it must exist within itself.
I say this not because I want a more detailed analysis but merely as an indication that the French are indeed very different from each other and how they differ greatly over what makes up their 'capital'. For Marx's theory was first conceived by Engels about two decades after he came across Theses on Liberty (1848), which were published only three years later [1904] – at least until his death! In

#### Reason
Unlike the latest ChatGPT versions, GPT2 is an unaligned, non instruction tuned LLM. It was trained to continue texts, not to answer questions. It needs to be fine tuned to get better results as output.

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

args = TrainingArguments(
    output_dir="llama2_medical_lora",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=100,
    save_steps=500,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tokenized,
)
trainer.train()

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Step,Training Loss
100,2.069000
200,1.429500
300,1.345200
400,1.316800
500,1.330200
600,1.308300
700,1.283100
800,1.302700
900,1.287700
1000,1.331000


TrainOutput(global_step=1565, training_loss=1.3558445257119858, metrics={'train_runtime': 735.603, 'train_samples_per_second': 33.986, 'train_steps_per_second': 2.128, 'total_flos': 3311448883200000.0, 'train_loss': 1.3558445257119858, 'epoch': 5.0})

In [ ]:
model2 = PeftModel.from_pretrained(model, "/content/llama2_medical_lora/checkpoint-500")
model2.to(device)

outputs2 = model2.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=True,
    temperature=0.8,
    top_p=0.9,
    repetition_penalty=1.2
)
print(tokenizer.decode(outputs2[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is the capital of France?

### Response:
… [A man with white hair and blue eyes] The French have been very friendly toward us, as we are only three or four miles from our home town in Paris... (Note: I am not talking about any actual cities here.) We know they like to hear "the best" because this country seems so great all by itself! That's why you should be prepared for everything when going out on business today." Note: In case your phone was turned off during my interview at 9am Eastern


In [ ]:
print(model2.config)
print(f"Total parameters: {model2.num_parameters()/1e6:.1f}M")
ppl = calculate_perplexity(model2, tokenizer, train_data.select(range(200)))
print(f"Perplexity: {ppl:.2f}")

GPT2Config {
  "activation_function": "gelu_new",
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": 50256,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "transformers_version": "4.57.1",
  "use_cache": true,
  "vocab_size": 50257
}

Total parameters: 124.7M


100%|██████████| 200/200 [00:06<00:00, 28.90it/s]

Perplexity: 15.57
